In [1]:
from dotenv import load_dotenv
from openai import OpenAI
import json
import os
import requests
from pypdf import PdfReader
import gradio as gr

In [2]:
load_dotenv(override=True)
openai = OpenAI()

In [3]:
pushover_user = os.getenv("PUSHOVER_USER")
pushover_token = os.getenv("PUSHOVER_TOKEN")
pushover_url = "https://api.pushover.net/1/messages.json"

if pushover_user:
    print(f"Pushover user: {pushover_user}")
else:
    print("Pushover user not found")

if pushover_token:
    print(f"Pushover token: {pushover_token}")

Pushover user: ud98dk74put8mdbtdr1q3f9xhjcx6b
Pushover token: ayffc8pupbx3cfkzx26wdeue8stbvv


In [4]:
def push(message):
    payload = {
        "token": pushover_token,
        "user": pushover_user,
        "message": message
    }
    response = requests.post(pushover_url, data=payload)
    print (response)

In [5]:
def record_user_details_json (email, name="Not provided", notes ="not provided"):
    push(f"recording interest from {name} with email {email} and notes {notes}")
    return {"recorded":"ok"}

In [6]:
def record_unknown_question_json (question):
    push(f"recording unknown question: {question}")
    return {"recorded":"ok"}

In [24]:
record_user_details_json = {
    "name": "record_user_details_json",
    "description": "Use this tool to record that a user is interested in being in touch and provided an email address",
    "parameters": {
        "type": "object",
        "properties": {
            "email": {
                "type": "string", 
                "description": "The email address of this user"
            },
            "name": {
                "type": "string", 
                "description": "The user's name, if they provided it"
            },
            "notes": {
                "type": "string", 
                "description": "Any additional information about the conversation that's worth recording to give context"
            }
        },
        "required": ["email"],
        "additionalProperties": False
    }
}

In [ ]:
#print (type(record_user_details_json))

<class 'dict'>


In [29]:
record_unknown_question_json = {
    "name": "record_unknown_question_json",
    "description": "Always use this tool to record any question that couldn't be answered as you didn't know the answer",
    "parameters": {
        "type": "object",
        "properties": {
            "question": {
                "type": "string", 
                "description": "The question that couldn't be answered"
            },
        },
        "required": ["question"],
        "additionalProperties": False
    }
}

In [9]:
tools = [{"type": "function", "function": record_user_details_json}, 
        {"type": "function", "function": record_unknown_question_json}]        

In [ ]:
def handle_tool_calls(tool_calls):
    results = []
    for tool_call in tool_calls:
        tool_name = tool_call.function.name
        arguments = json.loads(tool_call.function.arguments)
        print(f"Tool called: {tool_name}", flush=True)

        if tool_name == "record_user_details_json":
             result = record_user_details_json(**arguments)
        elif tool_name == "record_unknown_question_json":
            result = record_unknown_question_json(**arguments)

        # if tool_name == "record_user_details_json":
        #     result = record_user_details_json(**arguments)
        # elif tool_name == "record_unknown_question_json":
        #     result = record_unknown_question_json(**arguments)

        results.append({"role": "tool","content": json.dumps(result),"tool_call_id": tool_call.id})

    return results

In [ ]:
#globals()["record_unknown_question_json"]("this is a really hard question")

TypeError: 'dict' object is not callable

In [13]:
# def handle_tool_calls(tool_calls):
#     results = []
#     for tool_call in tool_calls:
#         tool_name = tool_call.function.name
#         arguments = json.loads(tool_call.function.arguments)
#         print(f"Tool called: {tool_name}", flush=True)
#         tool = globals().get(tool_name)
#         result = tool(**arguments) if tool else {}
#         results.append({"role": "tool","content": json.dumps(result),"tool_call_id": tool_call.id})
#     return results

In [14]:
reader = PdfReader("me/linkedin.pdf")
linkedin = ""
for page in reader.pages:
    text = page.extract_text()
    if text:
        linkedin += text

with open("me/summary.txt", "r", encoding="utf-8") as f:
    summary = f.read()

name = "Javier Lopez"

In [15]:
system_prompt = f"You are acting as {name}. You are answering questions on {name}'s website, \
particularly questions related to {name}'s career, background, skills and experience. \
Your responsibility is to represent {name} for interactions on the website as faithfully as possible. \
You are given a summary of {name}'s background and LinkedIn profile which you can use to answer questions. \
Be professional and engaging, as if talking to a potential client or future employer who came across the website. \
If you don't know the answer to any question, use your record_unknown_question_json tool to record the question that you couldn't answer, even if it's about something trivial or unrelated to career. \
If the user is engaging in discussion, try to steer them towards getting in touch via email; ask for their email and record it using your record_user_details_json tool. "

system_prompt += f"\n\n## Summary:\n{summary}\n\n## LinkedIn Profile:\n{linkedin}\n\n"
system_prompt += f"With this context, please chat with the user, always staying in character as {name}."

In [16]:
def chat(message, history):
    messages = [{"role": "system", "content": system_prompt}] + history + [{"role": "user", "content": message}]
    done = False
    while not done:
        # This is the call to the LLM - see that we pass in the tools json
        response = openai.chat.completions.create(model="gpt-4.1-nano", messages=messages, tools=tools)
        finish_reason = response.choices[0].finish_reason
        # If the LLM wants to call a tool, we do that!
        if finish_reason=="tool_calls":
            message = response.choices[0].message
            tool_calls = message.tool_calls
            results = handle_tool_calls(tool_calls)
            messages.append(message)
            messages.extend(results)
        else:
            done = True
    return response.choices[0].message.content

In [17]:
gr.ChatInterface(chat, type="messages").launch()

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


Tool called: record_user_details_json


Traceback (most recent call last):
  File "/Users/fco.javierlopezabad/projects/agents/.venv/lib/python3.12/site-packages/gradio/queueing.py", line 759, in process_events
    response = await route_utils.call_process_api(
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/fco.javierlopezabad/projects/agents/.venv/lib/python3.12/site-packages/gradio/route_utils.py", line 354, in call_process_api
    output = await app.get_blocks().process_api(
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/fco.javierlopezabad/projects/agents/.venv/lib/python3.12/site-packages/gradio/blocks.py", line 2116, in process_api
    result = await self.call_function(
             ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/fco.javierlopezabad/projects/agents/.venv/lib/python3.12/site-packages/gradio/blocks.py", line 1621, in call_function
    prediction = await fn(*processed_input)
                 ^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/fco.javierlopezabad/projects/agents/.venv/lib